# 01. 전처리 — 원본 데이터 로딩 → pivot → 조인 → 캐시 저장

## 이 노트북의 목적 (Phase 0)

대회에서 주는 원본 데이터는 아래 두 가지 이유로 **그대로는 모델에 넣을 수 없는 형태**다.

1. **기상 예보(LDAPS, GFS)가 "긴 표(long format)"다.** 한 시각(`forecast_kst_dtm`)에 대해
   여러 개의 격자(grid)가 각각 한 행씩 차지하고 있다. 예를 들어 LDAPS는 시각 하나당 16개
   격자 행이 있다. 모델은 "시간 하나 = 행 하나"를 원하므로, 격자를 옆으로 펼쳐서
   컬럼으로 만드는 **pivot(피벗)** 작업이 필요하다. (pivot이란: 세로로 길게 쌓인 표를
   가로로 넓게 펼쳐서, 값이 있던 자리를 컬럼 이름으로 바꾸는 작업이다.)
2. **기상 데이터와 정답(발전량 라벨)이 서로 다른 파일에 있다.** 학습을 하려면 "이 시각의
   날씨 → 이 시각의 발전량"이 한 행에 같이 있어야 하므로, 시각을 기준으로 **조인(join,
   두 표를 공통 키로 이어 붙이는 것)** 해야 한다.

이 노트북은 위 두 작업(pivot, 조인)을 순서대로 진행하고, 그 결과를 `data/processed/`에
parquet(표를 압축해서 저장하는 파일 형식, csv보다 빠르고 용량이 작다) 파일로 저장한다.
이후 노트북들(`02_eda.ipynb` 이후)은 원본 CSV를 다시 읽지 않고 이 parquet 캐시를 읽는 것으로
시작한다.

추가로, SCADA(터빈에서 10분마다 수집한 실측 데이터) 실측치를 시간 단위로 합쳐서
`data/processed/scada_hourly.parquet`도 만든다. 이 데이터는 **test 기간에는 존재하지 않으므로
test 예측에 직접 쓸 수 없다** (CLAUDE.md 4번 규칙). 대신 학습 기간에서만 "예보 풍속 →
실제 발전량" 관계(파워커브)를 배우는 용도로 나중에(Phase 2) 쓸 것이다.

## 지킬 규칙
- 원본 CSV(`data/` 아래)는 절대 덮어쓰지 않는다. 결과는 전부 `data/processed/`에 새로 저장한다.
- 시간 관련 컬럼은 전부 KST 그대로 쓰고, tz 변환을 하지 않는다.
- LDAPS/GFS의 `data_available_kst_dtm`(그 예보를 실제로 쓸 수 있게 된 시각)은 그대로 보존해서,
  나중에 "예측기준시점 이전 정보만 사용했는가"를 코드로 확인할 수 있게 한다 (CLAUDE.md 4번,
  데이터 누수 금지 규칙).


In [1]:
# 이 프로젝트에서 공통으로 쓰는 경로 상수.
# CLAUDE.md는 data/ 밑이 평평한 구조라고 가정했지만, 실제로는 data/train/, data/test/
# 하위 폴더로 나뉘어 있었다 (data/ 폴더를 직접 열어 확인함). 아래 경로는 그 실제 구조를 따른다.
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 데이터 명세서(data_description.md)에 적힌 기간별 예상 행 수.
# 이후 셀에서 "로딩한 데이터가 예상한 만큼 있는가"를 확인하는 용도로 쓴다.
N_TRAIN_HOURS = 26304   # 2022-01-01 01:00 ~ 2025-01-01 00:00 (3년치 시간 수)
N_TEST_HOURS = 8760     # 2025-01-01 01:00 ~ 2026-01-01 00:00 (2025년 1년치 시간 수)
N_LDAPS_GRID = 16
N_GFS_GRID = 9

pd.set_option("display.max_columns", 10)


## 1. LDAPS 기상 예보 데이터

### 1-1. 원본(long format) 로딩

`ldaps_train.csv`를 그대로 읽어서 모양을 확인한다. 데이터 명세서 기준 예상 행 수는
`420,864 = 26,304시각 x 16격자`다. 실제로 그런지 셀 실행 결과로 확인한다.


In [2]:
ldaps_train_raw = pd.read_csv(
    TRAIN_DIR / "ldaps_train.csv",
    encoding="utf-8-sig",
    parse_dates=["forecast_kst_dtm", "data_available_kst_dtm"],
)

print("shape:", ldaps_train_raw.shape)
assert ldaps_train_raw.shape[0] == N_TRAIN_HOURS * N_LDAPS_GRID, "예상 행 수와 다릅니다"
ldaps_train_raw.head(3)


shape: (420864, 35)


,forecast_kst_dtm,data_available_kst_dtm,grid_id,latitude,longitude,...,surface_0_ncpcp,surface_0_snol,surface_0_SNOM,surface_0_lsm,surface_0_h
0,2022-01-01 01:00:00,2021-12-31 13:00:00,1,37.3032,128.9443,...,0.0,0.0,0.0,1.0,992.6250
1,2022-01-01 01:00:00,2021-12-31 13:00:00,2,37.3027,128.9617,...,0.0,0.0,0.0,1.0,936.5625
2,2022-01-01 01:00:00,2021-12-31 13:00:00,3,37.3022,128.9790,...,0.0,0.0,0.0,1.0,868.8125


### 1-2. pivot: long(격자가 행) → wide(격자가 컬럼)

**하는 일**: `forecast_kst_dtm`(예측 대상 시각)을 행으로 두고, 각 격자(`grid_id`)의
기상 변수들을 옆으로 펼쳐서 컬럼으로 만든다. 컬럼 이름은 `ldaps_g{격자번호}_{변수명}`
형태로 만든다 (예: `ldaps_g6_heightAboveGround_10_10u` = LDAPS 6번 격자의 지상 10m
U방향 바람).

**왜 이렇게 이름을 짓는가**: 나중에 GFS 데이터도 똑같이 pivot할 텐데, LDAPS와 GFS에
`surface_0_sp`, `meanSea_0_prmsl`처럼 **이름이 같은 변수**가 있다. 접두어(`ldaps_`,
`gfs_`)를 붙이지 않으면 두 표를 합칠 때 컬럼 이름이 충돌한다.

**데이터 형태 변화**: (420,864행 = 26,304시각 x 16격자, long) → (26,304행 x (30변수 x 16격자
+ 시각 관련 컬럼), wide). 격자별 위도/경도(`latitude`, `longitude`)는 시간에 따라 바뀌지
않는 고정값이라 pivot 대상에서 제외한다 (필요하면 `info.xlsx`나 별도 표로 참고).

이 함수는 GFS pivot에도 그대로 재사용할 것이므로 함수로 만든다 (같은 로직을 두 번
복사해서 쓰면, 나중에 한쪽만 고치는 실수가 나기 쉽다).


In [3]:
def pivot_weather_wide(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """
    기상 예보 데이터(long, 시각당 격자 여러 행)를 wide(시각당 한 행)로 바꾼다.

    입력:
        df    : forecast_kst_dtm, data_available_kst_dtm, grid_id, latitude, longitude
                + 기상 변수 컬럼들을 가진 DataFrame (long format).
        prefix: 결과 컬럼 이름 앞에 붙일 접두어. 예: "ldaps", "gfs".
                LDAPS와 GFS에 이름이 같은 변수(surface_0_sp 등)가 있어서,
                합칠 때 컬럼명이 겹치지 않도록 소스별로 구분해야 한다.

    출력:
        forecast_kst_dtm 1개 시각 = 1행인 DataFrame.
        컬럼: "forecast_kst_dtm", f"{prefix}_data_available_kst_dtm",
              f"{prefix}_g{grid_id}_{변수명}" (격자 x 변수 개수만큼).

    주의:
        - latitude/longitude는 격자별 고정값(시간에 따라 안 변함)이라 pivot에서 제외한다.
        - data_available_kst_dtm은 같은 forecast_kst_dtm이면 모든 격자에서 같은 값이어야
          정상이다(하루 24시간 단위로 같은 예보 시각에 공개되므로). 다르면 데이터 이상이므로
          assert로 걸러낸다.
        - 원본 자체에 특정 시각의 기상 변수가 통째로 빠져 있는 경우(예: ldaps_test.csv의
          3개 시각)가 있어서, pivot 후 시간순 선형보간으로 메운다. 이 보간은 "그 시각을
          기준으로 이미 공개된 같은 예보 자료 안에서" 앞뒤 값을 참고하는 것이라 미래의
          실제 발전량이나 다른 정보를 끌어오는 게 아니므로 CLAUDE.md 4번 누수 규칙과는
          무관하다(라벨을 참고하지 않음).
    """
    id_cols = {"forecast_kst_dtm", "data_available_kst_dtm", "grid_id", "latitude", "longitude"}
    met_cols = [c for c in df.columns if c not in id_cols]

    # grid_id별로 data_available_kst_dtm이 같은지 확인 (다르면 예보 공개 시각을 잘못 이해한 것)
    n_unique_avail = df.groupby("forecast_kst_dtm")["data_available_kst_dtm"].nunique()
    assert (n_unique_avail == 1).all(), "같은 forecast_kst_dtm인데 data_available_kst_dtm이 격자마다 다릅니다"

    # (시각 x 격자, long) -> (시각, wide): 격자(grid_id)를 컬럼으로 펼친다
    wide = df.pivot(index="forecast_kst_dtm", columns="grid_id", values=met_cols)
    wide.columns = [f"{prefix}_g{grid_id}_{var}" for var, grid_id in wide.columns]
    wide = wide.sort_index().reset_index()

    # data_available_kst_dtm은 격자마다 같으므로 대표값(첫 값) 하나만 붙인다
    avail = df.groupby("forecast_kst_dtm")["data_available_kst_dtm"].first()
    wide.insert(1, f"{prefix}_data_available_kst_dtm", wide["forecast_kst_dtm"].map(avail))

    # 원본 자체 결측 보간: 시간순으로 정렬돼 있으므로, 비어 있는 시각을 앞뒤 값으로 선형보간한다.
    met_wide_cols = [c for c in wide.columns if c.startswith(f"{prefix}_g")]
    n_missing = wide[met_wide_cols].isna().sum().sum()
    if n_missing > 0:
        missing_times = wide.loc[wide[met_wide_cols].isna().any(axis=1), "forecast_kst_dtm"].tolist()
        print(f"[{prefix}] 원본 결측 {n_missing}개 값 발견 (시각: {missing_times}) -> 선형보간으로 채움")
        wide[met_wide_cols] = wide[met_wide_cols].interpolate(method="linear", limit_direction="both")
    else:
        print(f"[{prefix}] 원본 결측 없음")

    return wide


ldaps_train_wide = pivot_weather_wide(ldaps_train_raw, prefix="ldaps")
print("shape:", ldaps_train_wide.shape)
assert ldaps_train_wide.shape[0] == N_TRAIN_HOURS
assert ldaps_train_wide.isna().sum().sum() == 0, "LDAPS pivot 결과에 결측이 있으면 안 됩니다"
ldaps_train_wide.iloc[:3, :5]


[ldaps] 원본 결측 없음
shape: (26304, 482)


,forecast_kst_dtm,ldaps_data_available_kst_dtm,ldaps_g1_heightAboveGround_10_10u,ldaps_g2_heightAboveGround_10_10u,ldaps_g3_heightAboveGround_10_10u
0,2022-01-01 01:00:00,2021-12-31 13:00:00,5.411171,5.593300,4.383828
1,2022-01-01 02:00:00,2021-12-31 13:00:00,4.968168,4.550687,3.019926
2,2022-01-01 03:00:00,2021-12-31 13:00:00,4.463653,4.102813,3.036407


### 1-3. LDAPS test에도 같은 함수 적용

train에서 만든 `pivot_weather_wide` 함수를 test에도 그대로 재사용한다. **train과 test가
완전히 같은 전처리 로직을 쓰도록 보장**하기 위해서다 (CLAUDE.md 12번 재현성 규칙). 예상
행 수는 `140,160 = 8,760시각 x 16격자`.

**주의 (실제로 실행해보고 발견한 것)**: 처음 이 함수를 test에 적용했을 때 결측이 있다는
assert가 실패했다. 원인을 찾아보니 `ldaps_test.csv` 원본 자체에 딱 3개 시각
(`2025-04-08 17:00`, `2025-06-18 18:00`, `2025-07-18 06:00`)에서 일부 기상 변수가
통째로 비어 있었다(16개 격자 전부). LDAPS train, GFS train, GFS test에는 결측이 전혀
없었으므로, LDAPS 기상청 자료 자체가 이 3개 시각에 일시적으로 끊긴 것으로 보인다.

이 3개 시각도 결국 발전량을 예측해서 제출해야 하는 시각이므로 행을 통째로 버릴 수 없다.
그래서 `pivot_weather_wide` 함수 안에 "시간순으로 정렬한 뒤 선형보간(앞뒤 값의 평균적인
흐름으로 빈 값을 채움)으로 메운다"는 처리를 추가했다. 결측이 없는 경우(train, GFS test)는
이 처리가 아무 일도 하지 않으므로, train과 test가 **완전히 같은 함수**를 타면서도 안전하다.


In [4]:
ldaps_test_raw = pd.read_csv(
    TEST_DIR / "ldaps_test.csv",
    encoding="utf-8-sig",
    parse_dates=["forecast_kst_dtm", "data_available_kst_dtm"],
)
assert ldaps_test_raw.shape[0] == N_TEST_HOURS * N_LDAPS_GRID

ldaps_test_wide = pivot_weather_wide(ldaps_test_raw, prefix="ldaps")
print("shape:", ldaps_test_wide.shape)
assert ldaps_test_wide.shape[0] == N_TEST_HOURS
assert ldaps_test_wide.isna().sum().sum() == 0
ldaps_test_wide.iloc[:3, :5]


[ldaps] 원본 결측 752개 값 발견 (시각: [Timestamp('2025-04-08 17:00:00'), Timestamp('2025-06-18 18:00:00'), Timestamp('2025-07-18 06:00:00')]) -> 선형보간으로 채움


shape: (8760, 482)


,forecast_kst_dtm,ldaps_data_available_kst_dtm,ldaps_g1_heightAboveGround_10_10u,ldaps_g2_heightAboveGround_10_10u,ldaps_g3_heightAboveGround_10_10u
0,2025-01-01 01:00:00,2024-12-31 13:00:00,6.154461,7.276531,7.838543
1,2025-01-01 02:00:00,2024-12-31 13:00:00,6.109220,7.313321,7.990567
2,2025-01-01 03:00:00,2024-12-31 13:00:00,6.497718,7.680823,8.340980


## 2. GFS 기상 예보 데이터

LDAPS와 원리는 같다 (long → wide). 격자 수만 9개로 다르고, 변수 이름도 다르다.
같은 `pivot_weather_wide` 함수를 `prefix="gfs"`로 재사용한다.


In [5]:
gfs_train_raw = pd.read_csv(
    TRAIN_DIR / "gfs_train.csv",
    encoding="utf-8-sig",
    parse_dates=["forecast_kst_dtm", "data_available_kst_dtm"],
)
assert gfs_train_raw.shape[0] == N_TRAIN_HOURS * N_GFS_GRID

gfs_train_wide = pivot_weather_wide(gfs_train_raw, prefix="gfs")
print("shape:", gfs_train_wide.shape)
assert gfs_train_wide.shape[0] == N_TRAIN_HOURS
assert gfs_train_wide.isna().sum().sum() == 0
gfs_train_wide.iloc[:3, :5]


[gfs] 원본 결측 없음
shape: (26304, 317)


,forecast_kst_dtm,gfs_data_available_kst_dtm,gfs_g1_heightAboveGround_10_10u,gfs_g2_heightAboveGround_10_10u,gfs_g3_heightAboveGround_10_10u
0,2022-01-01 01:00:00,2021-12-31 13:00:00,1.817466,2.947466,2.817466
1,2022-01-01 02:00:00,2021-12-31 13:00:00,1.781655,2.861655,2.851655
2,2022-01-01 03:00:00,2021-12-31 13:00:00,1.869741,2.819741,2.639741


In [6]:
gfs_test_raw = pd.read_csv(
    TEST_DIR / "gfs_test.csv",
    encoding="utf-8-sig",
    parse_dates=["forecast_kst_dtm", "data_available_kst_dtm"],
)
assert gfs_test_raw.shape[0] == N_TEST_HOURS * N_GFS_GRID

gfs_test_wide = pivot_weather_wide(gfs_test_raw, prefix="gfs")
print("shape:", gfs_test_wide.shape)
assert gfs_test_wide.shape[0] == N_TEST_HOURS
assert gfs_test_wide.isna().sum().sum() == 0
gfs_test_wide.iloc[:3, :5]


[gfs] 원본 결측 없음
shape: (8760, 317)


,forecast_kst_dtm,gfs_data_available_kst_dtm,gfs_g1_heightAboveGround_10_10u,gfs_g2_heightAboveGround_10_10u,gfs_g3_heightAboveGround_10_10u
0,2025-01-01 01:00:00,2024-12-31 13:00:00,2.142351,3.352351,4.782351
1,2025-01-01 02:00:00,2024-12-31 13:00:00,2.170967,3.210967,5.110967
2,2025-01-01 03:00:00,2024-12-31 13:00:00,2.291846,3.171846,4.861846


## 3. 실제 발전량 라벨(`train_labels.csv`)

라벨은 이미 시각당 1행(wide) 형태라 pivot이 필요 없다. 결측 현황만 확인한다.
CLAUDE.md 8번 규칙: `kpx_group_3`의 2022년 빈칸은 **0이 아니라 결측**이므로, 여기서
0으로 채우지 않고 NaN 그대로 둔다 (학습 단계에서 결측 행을 제외하거나, 그룹1·2로
사전학습 후 그룹3을 파인튜닝하는 전이학습을 Phase 3~4에서 검토할 것이다).


In [7]:
labels = pd.read_csv(
    TRAIN_DIR / "train_labels.csv",
    encoding="utf-8-sig",
    parse_dates=["kst_dtm"],
)
print("shape:", labels.shape)
assert labels.shape[0] == N_TRAIN_HOURS

print("\n컬럼별 결측 개수:")
print(labels.isna().sum())

# kpx_group_3 결측이 정확히 2022년 전체(8,760시간)와 일치하는지 확인
g3_missing_2022 = labels.loc[labels["kst_dtm"].dt.year == 2022, "kpx_group_3"].isna().sum()
g3_missing_total = labels["kpx_group_3"].isna().sum()
print(f"\n2022년 kpx_group_3 결측: {g3_missing_2022}개 (2022년 전체 시간 수: {(labels['kst_dtm'].dt.year==2022).sum()})")
print(f"전체 kpx_group_3 결측: {g3_missing_total}개 (2022년 외 산발적 결측: {g3_missing_total - g3_missing_2022}개)")
labels.head(3)


shape: (26304, 4)

컬럼별 결측 개수:
kst_dtm           0
kpx_group_1     104
kpx_group_2     103
kpx_group_3    8766
dtype: int64

2022년 kpx_group_3 결측: 8759개 (2022년 전체 시간 수: 8759)
전체 kpx_group_3 결측: 8766개 (2022년 외 산발적 결측: 7개)


,kst_dtm,kpx_group_1,kpx_group_2,kpx_group_3
0,2022-01-01 01:00:00,12004.421,9719.242,NaN
1,2022-01-01 02:00:00,12901.137,10297.768,NaN
2,2022-01-01 03:00:00,12091.200,10731.663,NaN


## 4. SCADA 실측 데이터 — 10분 → 1시간 집계, 라벨과 대조

### 4-1. 왜 이 작업을 하는가

SCADA는 터빈 한 대 한 대의 10분 단위 실측치(풍속, 풍향, 출력)다. 이 데이터 자체는
**test 기간(2025년)에 존재하지 않으므로 test 예측 입력으로 직접 쓸 수 없다** (CLAUDE.md
4번 규칙). 하지만 학습 기간에서는 다음 두 가지 용도로 쓸 수 있다 (CLAUDE.md 4번 허용 용법):

1. **파워커브 추정**: "터빈 풍속 → 터빈 출력" 관계를 학습해서, 나중에 예보 풍속만으로
   발전량을 물리적으로 추정하는 피처를 만든다 (Phase 2에서 사용).
2. **예보 풍속 → 실측 풍속 보정 모델 학습**: 예보가 실제 바람과 얼마나 다른지(편향)를
   학습 기간에서 배운다.

이 두 가지를 하려면 먼저 10분 단위 데이터를 **라벨과 같은 "1시간 단위, 종료 시각 기준"**
으로 맞춰야 한다. 그런데 컬럼 이름이 `power_kw10m`(10분 단위 kW)이라고 되어 있어서,
"평균 kW인지, 10분간 누적된 에너지(kWh)인지"가 이름만으로는 애매하다. 그래서 라벨과
직접 대조해서 단위를 확정한다 (CLAUDE.md 2번 규칙: "단위 해석은 코드로 라벨과
크로스체크 후 확정").

### 4-2. 두 가지 후보 가설과 검증 방법

- 가설 A: `power_kw10m`이 "그 10분 구간 동안의 평균 출력(kW)"이다. 이 경우 1시간
  발전량(kWh) = 6개 구간의 **평균** kW x 1시간 = 6개 값의 **평균**.
- 가설 B: `power_kw10m`이 이름과 달리 "그 10분 구간 동안 생산된 에너지"를 이미 kWh
  스케일로 담고 있다. 이 경우 1시간 발전량 = 6개 값의 **합**.

두 가설은 정확히 6배 차이가 나므로, 실제 라벨과 비교하면 바로 구분된다.


In [8]:
# 검증에는 라벨과 겹치는 학습 기간 SCADA 앞부분(하루치)만 쓴다 (검증 목적이므로 전체 로딩 불필요)
scada_probe = pd.read_csv(
    TRAIN_DIR / "scada_vestas_train.csv",
    encoding="utf-8-sig",
    nrows=200,
    parse_dates=["kst_dtm"],
).set_index("kst_dtm")

group1_power_cols = [f"vestas_wtg{i:02d}_power_kw10m" for i in range(1, 7)]

# 가설 A: 평균, 가설 B: 합. resample('1h', closed='right', label='right')는
# "(이전시각, 현재시각]" 구간을 현재시각 라벨로 묶는다 -- 라벨의 "구간 종료 시각" 규칙과 동일.
hourly_group_power = scada_probe[group1_power_cols].sum(axis=1)  # 시각별 6터빈 합
hyp_a_mean = hourly_group_power.resample("1h", closed="right", label="right").mean()
hyp_b_sum = hourly_group_power.resample("1h", closed="right", label="right").sum()

labels_idx = labels.set_index("kst_dtm")["kpx_group_1"]

compare = pd.DataFrame({
    "가설A_평균": hyp_a_mean,
    "가설B_합계": hyp_b_sum,
    "실제_라벨": labels_idx,
}).dropna().loc["2022-01-01 02:00:00":"2022-01-01 08:00:00"]
compare["가설A_오차율"] = (compare["가설A_평균"] - compare["실제_라벨"]).abs() / compare["실제_라벨"]
compare["가설B_오차율"] = (compare["가설B_합계"] - compare["실제_라벨"]).abs() / compare["실제_라벨"]
compare


,가설A_평균,가설B_합계,실제_라벨,가설A_오차율,가설B_오차율
kst_dtm,,,,,
2022-01-01 02:00:00,2181.666667,13090.0,12901.137,0.830893,0.014639
2022-01-01 03:00:00,2031.333333,12188.0,12091.200,0.831999,0.008006
2022-01-01 04:00:00,2915.666667,17494.0,17167.768,0.830166,0.019003
2022-01-01 05:00:00,3231.166667,19387.0,19134.758,0.831136,0.013182
2022-01-01 06:00:00,2833.833333,17003.0,16806.189,0.831382,0.011711
2022-01-01 07:00:00,2033.000000,12198.0,12177.979,0.833059,0.001644
2022-01-01 08:00:00,1512.000000,9072.0,9111.789,0.834061,0.004367


**결과 해석**: 위 표에서 "가설B_오차율"이 "가설A_오차율"보다 훨씬 작다 (몇 % 이내).
즉 **`power_kw10m`은 이미 10분 구간의 에너지(kWh 스케일)를 담고 있고, 1시간 발전량은
6개 구간을 그냥 합(sum)하면 된다.** CLAUDE.md 2번 규칙이 제안했던 "합산 후 /6" 방식은
가설 A(평균)에 해당하는데, 실제로는 나누지 않는 가설 B가 라벨과 훨씬 잘 맞는다는 것을
이 검증으로 확인했다. 이후 모든 SCADA 시간 집계는 **합(sum)** 방식을 쓴다.

### 4-3. 전체 기간 집계 함수 작성

이제 전체 SCADA 데이터를 시간 단위로 집계한다. 출력(`power_kw10m`)은 **합**, 풍속·풍향은
**평균**으로 집계한다 (풍속/풍향은 "누적"이 의미 없는 순간값이므로 평균이 자연스럽다).


In [9]:
def aggregate_scada_hourly(df: pd.DataFrame, turbine_ids: list) -> pd.DataFrame:
    """
    10분 단위 SCADA 데이터를 1시간 단위로 집계한다.

    입력:
        df         : kst_dtm 인덱스(10분 간격)를 가진 DataFrame.
                     {turbine}_power_kw10m, {turbine}_ws, {turbine}_wd 컬럼을 포함해야 함.
        turbine_ids: 터빈 식별자 리스트. 예: ["vestas_wtg01", ..., "vestas_wtg12"].

    출력:
        1시간 간격 DataFrame. 컬럼:
            - {turbine}_power_kwh: 그 시간의 발전량 합계 (kWh). 6개 10분 구간의 합.
              (4-2 검증에서 확인한 대로 "합"이 맞는 단위 -- 평균이 아님)
            - {turbine}_ws_mean  : 그 시간의 평균 풍속
            - {turbine}_wd_mean  : 그 시간의 평균 풍향
            - n_samples          : 그 시간에 실제로 존재한 10분 샘플 개수 (정상이면 6).
                                    6보다 작으면 결측/누락이 있었다는 뜻이므로 남겨서
                                    나중에 신뢰도 판단에 쓴다.
    """
    resampled = df.resample("1h", closed="right", label="right")

    out = pd.DataFrame(index=resampled.size().index)
    out["n_samples"] = resampled.size()
    for tid in turbine_ids:
        out[f"{tid}_power_kwh"] = resampled[f"{tid}_power_kw10m"].sum()
        out[f"{tid}_ws_mean"] = resampled[f"{tid}_ws"].mean()
        out[f"{tid}_wd_mean"] = resampled[f"{tid}_wd"].mean()
    return out.reset_index()


vestas_ids = [f"vestas_wtg{i:02d}" for i in range(1, 13)]
scada_vestas_raw = pd.read_csv(TRAIN_DIR / "scada_vestas_train.csv", encoding="utf-8-sig", parse_dates=["kst_dtm"]).set_index("kst_dtm")
scada_vestas_hourly = aggregate_scada_hourly(scada_vestas_raw, vestas_ids)
print("shape:", scada_vestas_hourly.shape)
print("n_samples==6인 비율:", (scada_vestas_hourly["n_samples"] == 6).mean())
scada_vestas_hourly.iloc[:3, :5]


shape: (26304, 38)
n_samples==6인 비율: 0.9999619829683698


,kst_dtm,n_samples,vestas_wtg01_power_kwh,vestas_wtg01_ws_mean,vestas_wtg01_wd_mean
0,2022-01-01 01:00:00,1,325,8.500000,314.800000
1,2022-01-01 02:00:00,6,2279,9.083333,318.333333
2,2022-01-01 03:00:00,6,2132,8.916667,320.300000


In [10]:
unison_ids = [f"unison_wtg{i:02d}" for i in range(1, 6)]
scada_unison_raw = pd.read_csv(TRAIN_DIR / "scada_unison_train.csv", encoding="utf-8-sig", parse_dates=["kst_dtm"]).set_index("kst_dtm")
scada_unison_hourly = aggregate_scada_hourly(scada_unison_raw, unison_ids)
print("shape:", scada_unison_hourly.shape)
print("n_samples==6인 비율:", (scada_unison_hourly["n_samples"] == 6).mean())
scada_unison_hourly.iloc[:3, :5]


shape: (17544, 17)
n_samples==6인 비율: 1.0


,kst_dtm,n_samples,unison_wtg01_power_kwh,unison_wtg01_ws_mean,unison_wtg01_wd_mean
0,2023-01-01 01:00:00,6,3429.0,13.936667,-76.689105
1,2023-01-01 02:00:00,6,0.0,12.275000,-72.945694
2,2023-01-01 03:00:00,6,0.0,10.798333,-78.192769


### 4-4. 그룹 합계와 라벨 전체 기간 대조

앞서는 하루치만 봤다. 이번에는 **전체 학습 기간**에 대해 "SCADA 터빈 합계"와
"KPX 공식 라벨"의 비율(라벨 / SCADA합계)을 그룹별로 요약한다. 비율이 1에 가깝고
흩어짐(표준편차)이 작을수록 SCADA로 만든 피처를 믿을 수 있다는 뜻이다. 이 결과는
`reports/phase0_preprocessing.md`에 그대로 옮겨 "그룹3이 왜 더 어려운 그룹인지"를
설명하는 근거로 쓴다.


In [11]:
group1_kwh_cols = [f"{tid}_power_kwh" for tid in vestas_ids[0:6]]
group2_kwh_cols = [f"{tid}_power_kwh" for tid in vestas_ids[6:12]]
group3_kwh_cols = [f"{tid}_power_kwh" for tid in unison_ids]

scada_group_sum = pd.DataFrame({
    "kst_dtm": scada_vestas_hourly["kst_dtm"],
    "scada_group_1": scada_vestas_hourly[group1_kwh_cols].sum(axis=1),
    "scada_group_2": scada_vestas_hourly[group2_kwh_cols].sum(axis=1),
})
scada_group3 = pd.DataFrame({
    "kst_dtm": scada_unison_hourly["kst_dtm"],
    "scada_group_3": scada_unison_hourly[group3_kwh_cols].sum(axis=1),
})

check = labels.merge(scada_group_sum, on="kst_dtm", how="inner").merge(scada_group3, on="kst_dtm", how="left")

summary_rows = []
for g, scada_col in [(1, "scada_group_1"), (2, "scada_group_2"), (3, "scada_group_3")]:
    label_col = f"kpx_group_{g}"
    sub = check[[label_col, scada_col]].dropna()
    sub = sub[sub[scada_col] > 0]  # 0으로 나누기 방지 (정지 시간대 제외)
    ratio = sub[label_col] / sub[scada_col]
    summary_rows.append({
        "그룹": f"kpx_group_{g}",
        "대조 시간 수": len(sub),
        "비율 평균(라벨/SCADA)": ratio.mean(),
        "비율 표준편차": ratio.std(),
    })

ratio_summary = pd.DataFrame(summary_rows)
ratio_summary


,그룹,대조 시간 수,비율 평균(라벨/SCADA),비율 표준편차
0,kpx_group_1,22412,1.030412,0.926238
1,kpx_group_2,22423,1.031032,1.083050
2,kpx_group_3,14432,1.073688,1.102817


## 5. 학습용 데이터 병합 (LDAPS + GFS + 라벨)

이제 세 개의 wide 표(`ldaps_train_wide`, `gfs_train_wide`, `labels`)를 시각 기준으로
합친다. 세 표 모두 같은 기간(2022-01-01 01:00 ~ 2025-01-01 00:00)의 26,304개 시각을
가지고 있으므로, `forecast_kst_dtm`/`kst_dtm`을 공통 키로 삼아 안쪽 조인(inner join)해도
행이 줄어들면 안 된다 (줄어들면 세 표의 시각 범위가 어긋났다는 뜻이므로 assert로 확인).

LDAPS와 GFS 각각에 있는 `{prefix}_data_available_kst_dtm`을 서로 비교해서 값이 같은지도
확인한다. 두 예보 모두 "전날 09:00 초기화, 13:00부터 사용 가능"이라는 같은 규칙을 따르므로
같아야 정상이다. 같다면 최종 결과에는 하나만 `data_available_kst_dtm`으로 남긴다.


In [12]:
labels_for_merge = labels.rename(columns={"kst_dtm": "forecast_kst_dtm"})

train_merged = (
    ldaps_train_wide
    .merge(gfs_train_wide, on="forecast_kst_dtm", how="inner")
    .merge(labels_for_merge, on="forecast_kst_dtm", how="inner")
)
assert len(train_merged) == N_TRAIN_HOURS, "조인 후 행 수가 달라졌습니다 -- 세 표의 시각 범위를 확인하세요"

# LDAPS와 GFS의 data_available_kst_dtm이 서로 같은지 확인 후, 하나로 통합
avail_match = (train_merged["ldaps_data_available_kst_dtm"] == train_merged["gfs_data_available_kst_dtm"]).all()
print("LDAPS/GFS data_available_kst_dtm 일치 여부:", avail_match)
assert avail_match, "LDAPS와 GFS의 예보 공개 시각이 다릅니다 -- 원인을 확인해야 합니다"

train_merged = train_merged.rename(columns={"ldaps_data_available_kst_dtm": "data_available_kst_dtm"})
train_merged = train_merged.drop(columns=["gfs_data_available_kst_dtm"])

# 기상 피처(LDAPS/GFS)에는 결측이 없어야 한다. 라벨(kpx_group_3)만 2022년 결측이 있을 수 있다.
weather_cols = [c for c in train_merged.columns if c.startswith(("ldaps_g", "gfs_g"))]
assert train_merged[weather_cols].isna().sum().sum() == 0, "기상 피처에 결측이 있습니다"

print("최종 shape:", train_merged.shape)
train_merged[["forecast_kst_dtm", "data_available_kst_dtm", "kpx_group_1", "kpx_group_2", "kpx_group_3"]].head(3)


LDAPS/GFS data_available_kst_dtm 일치 여부: True
최종 shape: (26304, 800)


,forecast_kst_dtm,data_available_kst_dtm,kpx_group_1,kpx_group_2,kpx_group_3
0,2022-01-01 01:00:00,2021-12-31 13:00:00,12004.421,9719.242,NaN
1,2022-01-01 02:00:00,2021-12-31 13:00:00,12901.137,10297.768,NaN
2,2022-01-01 03:00:00,2021-12-31 13:00:00,12091.200,10731.663,NaN


In [13]:
out_path = PROCESSED_DIR / "train_merged.parquet"
train_merged.to_parquet(out_path, index=False)
print(f"저장 완료: {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)")


저장 완료: data\processed\train_merged.parquet (127.4 MB)


## 6. 평가용(test) 데이터 병합

test에는 라벨이 없으므로 LDAPS/GFS wide 표만 합친다. train과 **완전히 같은 함수
(`pivot_weather_wide`)와 같은 병합 로직**을 썼다는 점이 중요하다 -- train/inference 코드가
서로 다른 전처리를 쓰면 안 된다는 CLAUDE.md 12번 재현성 규칙을 지키기 위함이다.


In [14]:
test_merged = ldaps_test_wide.merge(gfs_test_wide, on="forecast_kst_dtm", how="inner")
assert len(test_merged) == N_TEST_HOURS

avail_match_test = (test_merged["ldaps_data_available_kst_dtm"] == test_merged["gfs_data_available_kst_dtm"]).all()
assert avail_match_test, "LDAPS와 GFS의 예보 공개 시각이 다릅니다"
test_merged = test_merged.rename(columns={"ldaps_data_available_kst_dtm": "data_available_kst_dtm"})
test_merged = test_merged.drop(columns=["gfs_data_available_kst_dtm"])

weather_cols_test = [c for c in test_merged.columns if c.startswith(("ldaps_g", "gfs_g"))]
assert test_merged[weather_cols_test].isna().sum().sum() == 0

# train과 test의 기상 피처 컬럼 집합이 완전히 같은지 확인 (다르면 이후 모델 입력이 어긋난다)
assert set(weather_cols_test) == set(weather_cols), "train/test 기상 피처 컬럼이 다릅니다"

print("최종 shape:", test_merged.shape)
test_merged[["forecast_kst_dtm", "data_available_kst_dtm"]].head(3)


최종 shape: (8760, 797)


,forecast_kst_dtm,data_available_kst_dtm
0,2025-01-01 01:00:00,2024-12-31 13:00:00
1,2025-01-01 02:00:00,2024-12-31 13:00:00
2,2025-01-01 03:00:00,2024-12-31 13:00:00


In [15]:
out_path_test = PROCESSED_DIR / "test_merged.parquet"
test_merged.to_parquet(out_path_test, index=False)
print(f"저장 완료: {out_path_test} ({out_path_test.stat().st_size / 1e6:.1f} MB)")


저장 완료: data\processed\test_merged.parquet (43.0 MB)


## 7. SCADA 시간 집계 캐시 저장

파워커브 추정, 예보 풍속 보정 모델 학습(Phase 2)에서 쓸 수 있도록 시간 단위로 집계한
SCADA 데이터도 캐시로 남긴다. **이 파일은 학습 기간에만 존재하고 test 기간에는 만들 수
없다** -- 그대로 test-time 피처로 merge하면 안 되고, "학습 데이터로 파워커브/보정계수를
학습 → 그 학습된 규칙(계수)만 test에 적용"하는 방식으로만 써야 한다 (CLAUDE.md 4번 규칙).


In [16]:
scada_vestas_hourly.to_parquet(PROCESSED_DIR / "scada_vestas_hourly.parquet", index=False)
scada_unison_hourly.to_parquet(PROCESSED_DIR / "scada_unison_hourly.parquet", index=False)
print("저장 완료:", PROCESSED_DIR / "scada_vestas_hourly.parquet")
print("저장 완료:", PROCESSED_DIR / "scada_unison_hourly.parquet")


저장 완료: data\processed\scada_vestas_hourly.parquet
저장 완료: data\processed\scada_unison_hourly.parquet


## 8. 최종 검증 (캐시를 다시 읽어서 확인)

"Restart & Run All"로 이 노트북을 처음부터 다시 실행해도 문제없이 재현된다는 것을
보여주기 위해, 방금 저장한 parquet 파일들을 새로 읽어와 shape과 기간을 다시 확인한다.
이 셀이 통과하면 `02_eda.ipynb`부터는 이 parquet만 읽으면 된다.


In [17]:
check_train = pd.read_parquet(PROCESSED_DIR / "train_merged.parquet")
check_test = pd.read_parquet(PROCESSED_DIR / "test_merged.parquet")

print("train_merged:", check_train.shape, check_train["forecast_kst_dtm"].min(), "~", check_train["forecast_kst_dtm"].max())
print("test_merged :", check_test.shape, check_test["forecast_kst_dtm"].min(), "~", check_test["forecast_kst_dtm"].max())
print("\ntrain_merged 컬럼 수:", len(check_train.columns), "| test_merged 컬럼 수:", len(check_test.columns))
print("(차이 3 = train에만 있는 kpx_group_1/2/3)")


train_merged: (26304, 800) 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00
test_merged : (8760, 797) 2025-01-01 01:00:00 ~ 2026-01-01 00:00:00

train_merged 컬럼 수: 800 | test_merged 컬럼 수: 797
(차이 3 = train에만 있는 kpx_group_1/2/3)
